# Iteración V2: XGBoost Avanzado (TimeSeriesSplit & Rolling Features)

En lugar de depender únicamente del mes anterior (Lag 1) y hacer un corte estático de 80/20, vamos a:
1. **Crear Rolling Averages:** Suavizar el ruido calculando la media de gastos e ingresos de los últimos 3 meses.
2. **Diferenciales:** Ver cómo cambia el ingreso mes a mes (e.g. ¿He cobrado más este mes?).
3. **Validación Cruzada Temporal (`TimeSeriesSplit`):** Entrenar el modelo iterativamente simulando el paso del tiempo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

### 1. Carga, Limpieza y Agrupación Mensual

In [ ]:
# Cargar datos
file_path = '../data/raw/db_orig.csv'
df = pd.read_csv(file_path)

# Limpieza
df['Amount'] = df['Amount'].str.replace('€', '', regex=False)
df['Amount'] = df['Amount'].str.replace('.', '', regex=False)
df['Amount'] = df['Amount'].str.replace(',', '.', regex=False)
df['Amount'] = pd.to_numeric(df['Amount'])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

# Agrupar por mes
df['Month_End'] = df['Date'].dt.to_period('M').dt.to_timestamp('M')
monthly = df.groupby(['Month_End', 'Type'])['Amount'].sum().unstack(fill_value=0).reset_index()

if 'Expenses' not in monthly.columns: monthly['Expenses'] = 0
if 'Income' not in monthly.columns: monthly['Income'] = 0

ml_df = monthly.copy().sort_values('Month_End')

### 2. Feature Engineering Avanzado

In [ ]:
### Básico
ml_df['Month'] = ml_df['Month_End'].dt.month
ml_df['Is_Summer'] = ml_df['Month'].isin([6, 7, 8, 9]).astype(int)

### Lags
ml_df['Income_Lag_1'] = ml_df['Income'].shift(1)
ml_df['Expenses_Lag_1'] = ml_df['Expenses'].shift(1)
ml_df['Expenses_Lag_2'] = ml_df['Expenses'].shift(2)

### Novedad: Rolling Averages (Promedio Móvil de 3 meses)
# Nos da una idea del "nivel de vida" reciente
ml_df['Expenses_Rolling_3'] = ml_df['Expenses'].shift(1).rolling(window=3).mean()
ml_df['Income_Rolling_3'] = ml_df['Income'].shift(1).rolling(window=3).mean()

### Novedad: Cambios Mensuales (Diferencias)
# Diferencia de ingresos respecto al mes anterior
ml_df['Income_Diff'] = ml_df['Income_Lag_1'] - ml_df['Income'].shift(2)

# Eliminar NaNs
ml_df = ml_df.dropna().reset_index(drop=True)
display(ml_df[['Month_End', 'Expenses', 'Expenses_Rolling_3', 'Income_Diff']].head())

### 3. Entrenamiento con TimeSeriesSplit y GridSearch
Vamos a probar distintos parámetros para XGBoost y los validaremos usando ventanas temporales deslizantes, que es la forma matemáticamente estricta de evaluar series temporales.

In [ ]:
FEATURES = ['Month', 'Is_Summer', 'Income', 'Income_Lag_1', 'Expenses_Lag_1', 
            'Expenses_Lag_2', 'Expenses_Rolling_3', 'Income_Rolling_3', 'Income_Diff']
TARGET = 'Expenses'

X = ml_df[FEATURES]
y = ml_df[TARGET]

# Configurar Validación Cruzada Temporal (3 particiones)
tscv = TimeSeriesSplit(n_splits=3)

# Parámetros a probar (GridSearch)
param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [2, 3, 5]
}

# Modelo Base
base_reg = xgb.XGBRegressor(objective='reg:squarederror')

# Búsqueda
grid_search = GridSearchCV(estimator=base_reg, param_grid=param_grid, 
                           cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)

# Usamos un 80% del set completo para hacer la búsqueda
split_idx = int(len(ml_df) * 0.8)
X_train, y_train = X.iloc[:split_idx], y.iloc[:split_idx]
X_test, y_test = X.iloc[split_idx:], y.iloc[split_idx:]

grid_search.fit(X_train, y_train)

print("Mejores Hiperparámetros:", grid_search.best_params_)
best_xgb = grid_search.best_estimator_

### 4. Evaluación Final

In [ ]:
# Predecir en el 20% final de pura prueba
predictions = best_xgb.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"XGBoost V2 - MAE: {mae:.2f}€")
print(f"XGBoost V2 - RMSE: {rmse:.2f}€")

xgb.plot_importance(best_xgb, importance_type='gain', max_num_features=5, title='Top 5 Variables (Ganancia)')
plt.show()